We want to study the abilities of GPT-1 pretrained to generate meaningful general representations. 
In order to do that, we plan to : 

- 1) Select many differents tasks and datasets 
- 2) Load GPT-1 pretrained state 
- 3) For each task, for $0 \leq k \leq 12$, we freeze the weights of the first $k$ layers, and we fine-tune on specific task 
- 4) We compare the ability to have a good performance while limiting the number of fine tuned parameters

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, OpenAIGPTModel
import numpy as np 
import torch 
import matplotlib.pyplot as plt

In [ ]:
# Load Data sets

 
dataset = load_dataset("stanfordnlp/sst2")

train_ds = dataset["train"].shuffle(seed=43).select(range(5000))
eval_ds = dataset["validation"]

print("train size : ", len(train_ds))
print("test size : ", len(eval_ds))
print("example : ", train_ds[0])

In [5]:
# Load pretrained checkpoint 
model_id = "openai-community/openai-gpt"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = OpenAIGPTModel.from_pretrained(model_id)
model.eval()               

print("embedding dim : ", model.config.n_embd)   

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 17157.98it/s]
[transformers] OpenAIGPTModel LOAD REPORT from: openai-community/openai-gpt
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dim :  768


In [ ]:
SEED = 42
MAX_TRAIN_SIZE = 5000
MAX_EVAL_SIZE = 1000

# Colonnes contenant les textes à donner au modèle.
TEXT_COLUMNS = {
    "sst2": ("sentence",),
    "cola": ("sentence",),
    "mrpc": ("sentence1", "sentence2"),
    "qqp":  ("question1", "question2"),
    "rte":  ("sentence1", "sentence2"),
    "qnli": ("question", "sentence"),
    "mnli": ("premise", "hypothesis"),
}

benchmarks = {}

for name, text_columns in TEXT_COLUMNS.items():
    raw = load_dataset("nyu-mll/glue", name)

    eval_split = (
        "validation_matched" if name == "mnli" else "validation"
    )

    train_ds = raw["train"]
    eval_ds = raw[eval_split]

    if MAX_TRAIN_SIZE is not None:
        train_ds = train_ds.shuffle(seed=SEED).select(
            range(min(MAX_TRAIN_SIZE, len(train_ds)))
        )

    if MAX_EVAL_SIZE is not None:
        eval_ds = eval_ds.shuffle(seed=SEED).select(
            range(min(MAX_EVAL_SIZE, len(eval_ds)))
        )

    label_feature = raw["train"].features["label"]

    benchmarks[name] = {
        "train": train_ds,
        "eval": eval_ds,
        "text_columns": text_columns,
        "num_labels": label_feature.num_classes,
        "label_names": label_feature.names,
    }

    print(
        f"{name.upper():5s} | "
        f"train={len(train_ds):5d} | "
        f"eval={len(eval_ds):4d} | "
        f"classes={label_feature.num_classes}"
    )